In [136]:
import numpy as np

from numba import njit
from numpy import ndarray

In [137]:
records = 15
rewards = np.random.randn(records, 32, 1)
values = np.random.randn(records, 32, 1)
episodes = [x//5 for x in range(records)]

In [138]:
def calculate(rewards: ndarray, values: ndarray, episodes, gamma=0.95, gae_lambda=0.91, alpha=1):
    steps = len(rewards)
    advantages          = []
    returns             = []
    future_value: ndarray = np.zeros_like(rewards[0])
    future_advantage: ndarray = np.zeros_like(rewards[0])
    prev_ep_idx         = episodes[-1]
    factor              = len(np.unique(episodes))-1
    for step in range(steps-1, -1, -1):
        reward, value, ep_idx = rewards[step], values[step], episodes[step]
        if prev_ep_idx != ep_idx:
            future_value = np.zeros_like(rewards[0])
            future_advantage = np.zeros_like(rewards[0])
            factor -= 1
        delta = reward + gamma*future_value - value
        future_advantage = delta + gamma*gae_lambda*future_advantage
        future_value = value
        advantage = future_advantage * ((1+alpha) ** factor)
        advantages.insert(0, advantage)
        returns.insert(0, advantage + value)
        prev_ep_idx = ep_idx
    return np.stack(returns), np.stack(advantages)

In [139]:
# %%timeit -n 10 -r 3
returns, advantages = calculate(rewards, values, episodes, gae_lambda=0.91)

In [140]:
rewards[:, 0].T

array([[-0.49925471,  0.88277999, -0.02027989, -0.46430032, -0.64518654,
         0.10260697, -1.98135184,  1.59699397, -4.09806711,  0.33417062,
        -1.23423239, -0.66374715,  0.21455917,  0.77479881, -0.47260098]])

In [141]:
values[:, 0].T

array([[ 0.44188169, -0.24055344, -1.22852611,  0.62395738,  0.36838926,
        -0.29128368,  0.16883719, -0.83122122,  0.72173096,  0.39597421,
        -0.58169041, -0.36043278,  0.57765734, -0.20043545,  0.72688664]])

In [142]:
returns[:, 0].T

array([[-0.46274909,  0.06601844, -0.82327654, -0.9905668 , -0.64518654,
        -5.42238797, -7.04883342, -2.37890454, -8.27237258,  0.27236704,
        -3.74228945,  0.58718255,  0.53806787,  2.31484267, -4.07106381]])

In [143]:
returns, _ = calculate(rewards, values, episodes, gae_lambda=0)
returns[:, 0].T

array([[-0.72778048, -0.28431981,  0.57247962, -0.11433053, -0.64518654,
         0.81728828, -5.71086118,  5.39649797, -8.16551418,  0.27236704,
        -4.5615029 ,  0.62140765, -1.63639007,  6.46267083, -4.07106381]])